## 13_param_task — Job task receiving job parameters (practice-job-3-parameters)

The job passes: environment, sample_rows, SCHEMA, PARTITION, CONFIG_PATH.
Manual-run fallback defaults are provided so it also runs standalone.
PARTITION: all | latest | prev | YYYY-MM-DD


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")
dbutils.widgets.text("environment", "practice")
dbutils.widgets.text("sample_rows", "10")

environment = dbutils.widgets.get("environment").strip()
sample_rows = int(dbutils.widgets.get("sample_rows"))

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

from pyspark.sql import functions as F

print(f"PARAM environment = {environment}")
print(f"PARAM sample_rows = {sample_rows}")
print(f"PARAM PARTITION   = {CFG.run_date}")

src = CFG.table("practice_customers_clean")
df = CFG.partition_filter(spark.read.table(src), table=src).limit(sample_rows)
display(df)
print(f"PARAM TASK DONE — sampled {df.count()} rows for environment '{environment}'")
